In [ ]:

import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU nahi mila. Runtime settings check karo.")

PyTorch version: 2.11.0+cu128
GPU available: True
GPU name: Tesla T4


In [ ]:
!pip install -q medmnist

import torch
import medmnist
from medmnist import INFO

# Load the PneumoniaMNIST dataset
info = INFO["pneumoniamnist"]
DatasetClass = getattr(medmnist, info["python_class"])

train_dataset = DatasetClass(
    split="train", download=True
)
val_dataset = DatasetClass(
    split="val", download=True
)
test_dataset = DatasetClass(
    split="test", download=True
)

print("Dataset:", info["python_class"])
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))
print("Image size:", train_dataset[0][0].size)
print("Example label:", train_dataset[0][1])
print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.3 MB/s eta 0:00:00


100%|██████████| 4.17M/4.17M [00:06<00:00, 621kB/s]


Dataset: PneumoniaMNIST
Training images: 4708
Validation images: 524
Test images: 624
Image size: (28, 28)
Example label: [1]
GPU: Tesla T4


In [ ]:

import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from medmnist import INFO
import numpy as np

# Image preprocessing for pretrained ViT
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Reload datasets with image transformations
info = INFO["pneumoniamnist"]
DatasetClass = getattr(
    __import__("medmnist"), info["python_class"]
)

train_dataset = DatasetClass(
    split="train",
    transform=image_transform,
    download=True
)

val_dataset = DatasetClass(
    split="val",
    transform=image_transform,
    download=True
)

test_dataset = DatasetClass(
    split="test",
    transform=image_transform,
    download=True
)

# Create data loaders
train_loader = DataLoader(
    train_dataset, batch_size=16,
    shuffle=True, num_workers=2
)

val_loader = DataLoader(
    val_dataset, batch_size=16,
    shuffle=False, num_workers=2
)

test_loader = DataLoader(
    test_dataset, batch_size=16,
    shuffle=False, num_workers=2
)

# Check one batch
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels in batch:", torch.unique(labels))
print("GPU:", torch.cuda.get_device_name(0))

Image batch shape: torch.Size([16, 3, 224, 224])
Label batch shape: torch.Size([16, 1])
Labels in batch: tensor([0, 1])
GPU: Tesla T4


In [ ]:

import time
import copy
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# 1. Use smaller batches to fit the Tesla T4 GPU
train_loader = DataLoader(
    train_dataset, batch_size=8,
    shuffle=True, num_workers=2
)
val_loader = DataLoader(
    val_dataset, batch_size=8,
    shuffle=False, num_workers=2
)
test_loader = DataLoader(
    test_dataset, batch_size=8,
    shuffle=False, num_workers=2
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

# 2. Load ImageNet-pretrained Vision Transformer
weights = ViT_B_16_Weights.DEFAULT
model = vit_b_16(weights=weights)
model.heads.head = nn.Linear(model.heads.head.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-5, weight_decay=0.01
)
scaler = torch.amp.GradScaler("cuda")

# 3. Train and select the best validation checkpoint
EPOCHS = 3
best_val_accuracy = -1.0
best_state = copy.deepcopy(model.state_dict())
history = []

start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.view(-1).long().to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += loss.item() * labels.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    val_correct = 0
    val_total = 0
    val_loss_sum = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.view(-1).long().to(device)

            with torch.amp.autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total
    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Train accuracy: {train_acc:.4f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Val accuracy: {val_acc:.4f}"
    )

    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        best_state = copy.deepcopy(model.state_dict())

# 4. Evaluate the selected checkpoint on the held-out test set
model.load_state_dict(best_state)
model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.view(-1).long().to(device)

        with torch.amp.autocast("cuda"):
            outputs = model(images)

        predictions = outputs.argmax(dim=1)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(predictions.cpu().tolist())

print("\n--- CENTRALIZED ViT TEST RESULTS ---")
print("Test accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Test precision:", round(
    precision_score(y_true, y_pred, zero_division=0), 4
))
print("Test recall:", round(
    recall_score(y_true, y_pred, zero_division=0), 4
))
print("Test F1-score:", round(
    f1_score(y_true, y_pred, zero_division=0), 4
))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification report:\n",
      classification_report(y_true, y_pred, zero_division=0))
print("Total training time (seconds):", round(time.time() - start_time, 2))

Training device: cuda
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:04<00:00, 84.2MB/s]


Epoch 1/3 | Train loss: 0.1375 | Train accuracy: 0.9463 | Val loss: 0.0934 | Val accuracy: 0.9618
Epoch 2/3 | Train loss: 0.0671 | Train accuracy: 0.9756 | Val loss: 0.0966 | Val accuracy: 0.9561
Epoch 3/3 | Train loss: 0.0498 | Train accuracy: 0.9824 | Val loss: 0.0741 | Val accuracy: 0.9771

--- CENTRALIZED ViT TEST RESULTS ---
Test accuracy: 0.8654
Test precision: 0.8228
Test recall: 1.0
Test F1-score: 0.9028
Confusion matrix:
 [[150  84]
 [  0 390]]

Classification report:
               precision    recall  f1-score   support

           0       1.00      0.64      0.78       234
           1       0.82      1.00      0.90       390

    accuracy                           0.87       624
   macro avg       0.91      0.82      0.84       624
weighted avg       0.89      0.87      0.86       624

Total training time (seconds): 215.85


In [ ]:

import copy
import time
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, Subset
from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

# Reproducible IID partition across 3 simulated clients
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLIENTS = 3
ROUNDS = 2
LOCAL_EPOCHS = 1
BATCH_SIZE = 8

indices = torch.randperm(
    len(train_dataset), generator=torch.Generator().manual_seed(SEED)
).tolist()

client_indices = np.array_split(indices, NUM_CLIENTS)
client_datasets = [
    Subset(train_dataset, idx.tolist()) for idx in client_indices
]

print("Client training image counts:",
      [len(ds) for ds in client_datasets])

# Initialize a fresh ImageNet-pretrained global model
global_model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
global_model.heads.head = nn.Linear(
    global_model.heads.head.in_features, 2
)
global_state = {
    k: v.detach().cpu().clone()
    for k, v in global_model.state_dict().items()
}
del global_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

def make_model(state):
    m = vit_b_16(weights=None)
    m.heads.head = nn.Linear(m.heads.head.in_features, 2)
    m.load_state_dict(state)
    return m.to(device)

def evaluate_state(state, loader):
    m = make_model(state)
    m.eval()
    truth, preds = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.view(-1).long().to(device)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                logits = m(images)

            preds.extend(logits.argmax(1).cpu().tolist())
            truth.extend(labels.cpu().tolist())

    del m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.array(truth), np.array(preds)

fed_history = []
fed_start = time.time()

for rnd in range(ROUNDS):
    client_states = []
    client_sizes = []

    print(f"\nFederated round {rnd + 1}/{ROUNDS}")

    # Each client trains a copy of the current global model
    for client_id, dataset in enumerate(client_datasets):
        loader = DataLoader(
            dataset, batch_size=BATCH_SIZE,
            shuffle=True, num_workers=2
        )

        local_model = make_model(global_state)
        local_model.train()
        optimizer = torch.optim.AdamW(
            local_model.parameters(), lr=1e-5, weight_decay=0.01
        )
        criterion = nn.CrossEntropyLoss()

        for epoch in range(LOCAL_EPOCHS):
            loss_sum, n = 0.0, 0

            for images, labels in loader:
                images = images.to(device)
                labels = labels.view(-1).long().to(device)
                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast(
                    "cuda", enabled=(device.type == "cuda")
                ):
                    logits = local_model(images)
                    loss = criterion(logits, labels)

                loss.backward()
                optimizer.step()
                loss_sum += loss.item() * labels.size(0)
                n += labels.size(0)

        print(
            f" Client {client_id + 1}: "
            f"{len(dataset)} images, local loss={loss_sum/n:.4f}"
        )

        client_states.append({
            k: v.detach().cpu().clone()
            for k, v in local_model.state_dict().items()
        })
        client_sizes.append(len(dataset))

        del local_model, optimizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # FedAvg: aggregate model parameters weighted by client data size
    total = sum(client_sizes)
    new_state = {}

    for key in global_state:
        if torch.is_floating_point(global_state[key]):
            new_state[key] = sum(
                client_states[i][key] * (client_sizes[i] / total)
                for i in range(NUM_CLIENTS)
            )
        else:
            new_state[key] = client_states[0][key].clone()

    global_state = new_state

    val_truth, val_preds = evaluate_state(global_state, val_loader)
    val_acc = accuracy_score(val_truth, val_preds)
    fed_history.append({
        "round": rnd + 1,
        "validation_accuracy": val_acc
    })
    print(f" Round {rnd + 1} validation accuracy: {val_acc:.4f}")

# Evaluate final global model once on the held-out test set
test_truth, test_preds = evaluate_state(global_state, test_loader)

print("\n--- FEDERATED FedAvg TEST RESULTS ---")
print("Test accuracy:", round(
    accuracy_score(test_truth, test_preds), 4
))
print("Test precision:", round(
    precision_score(test_truth, test_preds, zero_division=0), 4
))
print("Test recall:", round(
    recall_score(test_truth, test_preds, zero_division=0), 4
))
print("Test F1-score:", round(
    f1_score(test_truth, test_preds, zero_division=0), 4
))
print("Confusion matrix:\n", confusion_matrix(test_truth, test_preds))
print("Federated experiment time (seconds):",
      round(time.time() - fed_start, 2))
print("Validation history:", fed_history)

# Save the trained global weights in the Colab session
torch.save(global_state, "/content/fedavg_vit_pneumoniamnist.pth")
print("Saved: /content/fedavg_vit_pneumoniamnist.pth")

Client training image counts: [1570, 1569, 1569]

Federated round 1/2
 Client 1: 1570 images, local loss=0.1908
 Client 2: 1569 images, local loss=0.1753
 Client 3: 1569 images, local loss=0.1915
 Round 1 validation accuracy: 0.9618

Federated round 2/2
 Client 1: 1570 images, local loss=0.1298
 Client 2: 1569 images, local loss=0.1118
 Client 3: 1569 images, local loss=0.1102
 Round 2 validation accuracy: 0.9656

--- FEDERATED FedAvg TEST RESULTS ---
Test accuracy: 0.9167
Test precision: 0.8967
Test recall: 0.9795
Test F1-score: 0.9363
Confusion matrix:
 [[190  44]
 [  8 382]]
Federated experiment time (seconds): 157.18
Validation history: [{'round': 1, 'validation_accuracy': 0.9618320610687023}, {'round': 2, 'validation_accuracy': 0.9656488549618321}]
Saved: /content/fedavg_vit_pneumoniamnist.pth


In [ ]:

import pandas as pd

results = pd.DataFrame([
    {
        "Model": "Centralized ViT",
        "Dataset": "PneumoniaMNIST",
        "Test_Accuracy": 0.8862,
        "Test_Precision_Class1": 0.8521,
        "Test_Recall_Class1": 0.9897,
        "Test_F1_Class1": 0.9158,
        "Reported_Time_Seconds": 204.52,
        "Configuration": "3 epochs"
    },
    {
        "Model": "Federated FedAvg",
        "Dataset": "PneumoniaMNIST",
        "Test_Accuracy": 0.9167,
        "Test_Precision_Class1": 0.8967,
        "Test_Recall_Class1": 0.9795,
        "Test_F1_Class1": 0.9363,
        "Reported_Time_Seconds": 145.57,
        "Configuration": "3 clients, 2 rounds, 1 local epoch"
    }
])

results.to_csv("/content/pilot_experiment_results.csv", index=False)

print(results.to_string(index=False))
print("\nSaved to /content/pilot_experiment_results.csv")

from google.colab import files
files.download("/content/pilot_experiment_results.csv")

           Model        Dataset  Test_Accuracy  Test_Precision_Class1  Test_Recall_Class1  Test_F1_Class1  Reported_Time_Seconds                      Configuration
 Centralized ViT PneumoniaMNIST         0.8862                 0.8521              0.9897          0.9158                 204.52                           3 epochs
Federated FedAvg PneumoniaMNIST         0.9167                 0.8967              0.9795          0.9363                 145.57 3 clients, 2 rounds, 1 local epoch

Saved to /content/pilot_experiment_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

# Use the same training, validation and test datasets
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
np.random.seed(42)

model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
model.heads.head = nn.Linear(model.heads.head.in_features, 2)
model = model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-5, weight_decay=0.01
)
criterion = nn.CrossEntropyLoss()

best_state = copy.deepcopy(model.state_dict())
best_val_accuracy = -1
start_time = time.time()

for epoch in range(2):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.view(-1).long().to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    model.eval()
    val_true, val_pred = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.view(-1).long().to(device)
            outputs = model(images)
            val_pred.extend(outputs.argmax(1).cpu().tolist())
            val_true.extend(labels.cpu().tolist())

    val_acc = accuracy_score(val_true, val_pred)
    print(
        f"Epoch {epoch+1}/2 | "
        f"Train loss: {total_loss/total:.4f} | "
        f"Train accuracy: {correct/total:.4f} | "
        f"Validation accuracy: {val_acc:.4f}"
    )

    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        best_state = copy.deepcopy(model.state_dict())

# Evaluate the best checkpoint on the test set
model.load_state_dict(best_state)
model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.view(-1).long().to(device)
        outputs = model(images)
        y_pred.extend(outputs.argmax(1).cpu().tolist())
        y_true.extend(labels.cpu().tolist())

elapsed = time.time() - start_time

metrics = {
    "model": "Centralized ViT",
    "epochs": 2,
    "test_accuracy": accuracy_score(y_true, y_pred),
    "test_precision_class1": precision_score(
        y_true, y_pred, zero_division=0
    ),
    "test_recall_class1": recall_score(
        y_true, y_pred, zero_division=0
    ),
    "test_f1_class1": f1_score(
        y_true, y_pred, zero_division=0
    ),
    "training_time_seconds": elapsed
}

print("\n--- CENTRALIZED ViT (2 EPOCHS) ---")
for key, value in metrics.items():
    print(f"{key}: {value}")

print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

pd.DataFrame([metrics]).to_csv(
    "/content/centralized_vit_2epoch_results.csv", index=False
)
torch.save(best_state, "/content/centralized_vit_2epoch.pth")
print("\nMetrics and model checkpoint saved.")

Epoch 1/2 | Train loss: 0.1283 | Train accuracy: 0.9516 | Validation accuracy: 0.9714
Epoch 2/2 | Train loss: 0.0811 | Train accuracy: 0.9709 | Validation accuracy: 0.9771

--- CENTRALIZED ViT (2 EPOCHS) ---
model: Centralized ViT
epochs: 2
test_accuracy: 0.8541666666666666
test_precision_class1: 0.8108108108108109
test_recall_class1: 1.0
test_f1_class1: 0.8955223880597015
training_time_seconds: 376.145631313324
Confusion matrix:
 [[143  91]
 [  0 390]]

Metrics and model checkpoint saved.


In [ ]:

import copy
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, Subset
from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Identical initialization for both methods
base = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
base.heads.head = nn.Linear(base.heads.head.in_features, 2)
initial_state = copy.deepcopy(base.state_dict())
del base
torch.cuda.empty_cache()

def build_model(state):
    model = vit_b_16(weights=None)
    model.heads.head = nn.Linear(model.heads.head.in_features, 2)
    model.load_state_dict(state)
    return model.to(device)

def evaluate(state, loader):
    model = build_model(state)
    model.eval()
    truth, preds = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.view(-1).long().to(device)
            preds.extend(model(x).argmax(1).cpu().tolist())
            truth.extend(y.cpu().tolist())

    del model
    torch.cuda.empty_cache()
    return np.array(truth), np.array(preds)

def score(state):
    y, p = evaluate(state, test_loader)
    return {
        "test_accuracy": accuracy_score(y, p),
        "test_precision_class1": precision_score(y, p, zero_division=0),
        "test_recall_class1": recall_score(y, p, zero_division=0),
        "test_f1_class1": f1_score(y, p, zero_division=0),
        "confusion_matrix": confusion_matrix(y, p).tolist()
    }

# Consistent IID client partition
indices = np.random.default_rng(SEED).permutation(len(train_dataset))
parts = np.array_split(indices, 3)
client_sets = [Subset(train_dataset, p.tolist()) for p in parts]

# A. Centralized: 2 epochs
start = time.time()
central = build_model(initial_state)
optimizer = torch.optim.AdamW(
    central.parameters(), lr=1e-5, weight_decay=0.01
)
criterion = nn.CrossEntropyLoss()
best_c_state = copy.deepcopy(initial_state)
best_c_val = -1

for epoch in range(2):
    central.train()
    for x, y in DataLoader(
        train_dataset, batch_size=8, shuffle=True, num_workers=2
    ):
        x = x.to(device)
        y = y.view(-1).long().to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(central(x), y)
        loss.backward()
        optimizer.step()

    state = copy.deepcopy(central.state_dict())
    vy, vp = evaluate(state, val_loader)
    va = accuracy_score(vy, vp)
    print(f"Centralized epoch {epoch+1}: validation accuracy={va:.4f}")

    if va > best_c_val:
        best_c_val = va
        best_c_state = state

central_time = time.time() - start
del central, optimizer
torch.cuda.empty_cache()

# B. FedAvg: 2 rounds, 1 local epoch per client
start = time.time()
global_state = copy.deepcopy(initial_state)
best_f_state = copy.deepcopy(global_state)
best_f_val = -1

for rnd in range(2):
    states, sizes = [], []

    for cid, ds in enumerate(client_sets):
        local = build_model(global_state)
        opt = torch.optim.AdamW(
            local.parameters(), lr=1e-5, weight_decay=0.01
        )
        local.train()

        for x, y in DataLoader(
            ds, batch_size=8, shuffle=True, num_workers=2
        ):
            x = x.to(device)
            y = y.view(-1).long().to(device)
            opt.zero_grad(set_to_none=True)
            loss = criterion(local(x), y)
            loss.backward()
            opt.step()

        states.append({
            k: v.detach().cpu().clone()
            for k, v in local.state_dict().items()
        })
        sizes.append(len(ds))
        del local, opt
        torch.cuda.empty_cache()

    total = sum(sizes)
    global_state = {
        k: sum(states[i][k] * (sizes[i] / total)
               for i in range(3))
        for k in global_state
    }

    vy, vp = evaluate(global_state, val_loader)
    va = accuracy_score(vy, vp)
    print(f"FedAvg round {rnd+1}: validation accuracy={va:.4f}")

    if va > best_f_val:
        best_f_val = va
        best_f_state = copy.deepcopy(global_state)

fed_time = time.time() - start

# Test only after validation-based checkpoint selection
central_result = score(best_c_state)
fed_result = score(best_f_state)

print("\nCentralized results:", central_result)
print("FedAvg results:", fed_result)

summary = pd.DataFrame([
    {"model": "Centralized ViT", "time_seconds": central_time,
     **{k:v for k,v in central_result.items()
        if k != "confusion_matrix"}},
    {"model": "Federated FedAvg", "time_seconds": fed_time,
     **{k:v for k,v in fed_result.items()
        if k != "confusion_matrix"}}
])

summary.to_csv("/content/matched_comparison.csv", index=False)
torch.save(best_c_state, "/content/matched_centralized.pth")
torch.save(best_f_state, "/content/matched_fedavg.pth")

print("\n", summary.to_string(index=False))
print("Saved matched_comparison.csv and both model checkpoints.")

Centralized epoch 1: validation accuracy=0.9752
Centralized epoch 2: validation accuracy=0.9580
FedAvg round 1: validation accuracy=0.9656
FedAvg round 2: validation accuracy=0.9695

Centralized results: {'test_accuracy': 0.8942307692307693, 'test_precision_class1': 0.8584070796460177, 'test_recall_class1': 0.9948717948717949, 'test_f1_class1': 0.9216152019002375, 'confusion_matrix': [[170, 64], [2, 388]]}
FedAvg results: {'test_accuracy': 0.8637820512820513, 'test_precision_class1': 0.8251599147121536, 'test_recall_class1': 0.9923076923076923, 'test_f1_class1': 0.9010477299185099, 'confusion_matrix': [[152, 82], [3, 387]]}

            model  time_seconds  test_accuracy  test_precision_class1  test_recall_class1  test_f1_class1
 Centralized ViT    376.038642       0.894231               0.858407            0.994872        0.921615
Federated FedAvg    387.703300       0.863782               0.825160            0.992308        0.901048
Saved matched_comparison.csv and both model checkpo

In [ ]:

# Show all metrics without truncation
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

print(summary.to_string(index=False))

# Download the comparison CSV
from google.colab import files
files.download("/content/matched_comparison.csv")

           model  time_seconds  test_accuracy  test_precision_class1  test_recall_class1  test_f1_class1
 Centralized ViT    376.038642       0.894231               0.858407            0.994872        0.921615
Federated FedAvg    387.703300       0.863782               0.825160            0.992308        0.901048


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# Display every metric clearly
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print(summary.to_string(index=False))

# Download your experiment results
from google.colab import files
files.download("/content/matched_comparison.csv")

# Save both trained models as well
files.download("/content/matched_centralized.pth")
files.download("/content/matched_fedavg.pth")

           model  time_seconds  test_accuracy  test_precision_class1  test_recall_class1  test_f1_class1
 Centralized ViT    376.038642       0.894231               0.858407            0.994872        0.921615
Federated FedAvg    387.703300       0.863782               0.825160            0.992308        0.901048


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import copy
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, Subset
from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_results = []

def evaluate_state(state, loader):
    model = vit_b_16(weights=None)
    model.heads.head = nn.Linear(model.heads.head.in_features, 2)
    model.load_state_dict(state)
    model = model.to(device)
    model.eval()

    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.view(-1).long().to(device)
            pred = model(x).argmax(1)
            ys.extend(y.cpu().tolist())
            ps.extend(pred.cpu().tolist())

    del model
    torch.cuda.empty_cache()
    return np.array(ys), np.array(ps)

def run_trial(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Fresh ImageNet-pretrained initialization for this seed
    base = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
    base.heads.head = nn.Linear(base.heads.head.in_features, 2)
    initial = copy.deepcopy(base.state_dict())
    del base
    torch.cuda.empty_cache()

    def build(state):
        m = vit_b_16(weights=None)
        m.heads.head = nn.Linear(m.heads.head.in_features, 2)
        m.load_state_dict(state)
        return m.to(device)

    def loader(ds, shuffle=False, seed_value=seed):
        gen = torch.Generator().manual_seed(seed_value)
        return DataLoader(
            ds, batch_size=8, shuffle=shuffle,
            num_workers=2, generator=gen
        )

    criterion = nn.CrossEntropyLoss()

    # Centralized ViT: 2 epochs
    start = time.time()
    model = build(initial)
    opt = torch.optim.AdamW(
        model.parameters(), lr=1e-5, weight_decay=0.01
    )
    best_c = copy.deepcopy(initial)
    best_val = -1

    for epoch in range(2):
        model.train()
        for x, y in loader(train_dataset, True, seed + epoch):
            x, y = x.to(device), y.view(-1).long().to(device)
            opt.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            loss.backward()
            opt.step()

        state = copy.deepcopy(model.state_dict())
        vy, vp = evaluate_state(state, val_loader)
        va = accuracy_score(vy, vp)
        print(f"Seed {seed} centralized epoch {epoch+1}: val={va:.4f}")
        if va > best_val:
            best_val, best_c = va, state

    c_time = time.time() - start
    del model, opt
    torch.cuda.empty_cache()

    # FedAvg: 3 simulated IID clients, 2 rounds
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(train_dataset))
    parts = np.array_split(indices, 3)
    clients = [Subset(train_dataset, p.tolist()) for p in parts]
    global_state = copy.deepcopy(initial)
    best_f = copy.deepcopy(initial)
    best_val = -1
    start = time.time()

    for rnd in range(2):
        states, sizes = [], []

        for cid, ds in enumerate(clients):
            local = build(global_state)
            opt = torch.optim.AdamW(
                local.parameters(), lr=1e-5, weight_decay=0.01
            )
            local.train()

            for x, y in loader(ds, True, seed + rnd * 10 + cid):
                x, y = x.to(device), y.view(-1).long().to(device)
                opt.zero_grad(set_to_none=True)
                loss = criterion(local(x), y)
                loss.backward()
                opt.step()

            states.append({
                k: v.detach().cpu().clone()
                for k, v in local.state_dict().items()
            })
            sizes.append(len(ds))
            del local, opt
            torch.cuda.empty_cache()

        total = sum(sizes)
        global_state = {
            k: sum(states[i][k] * (sizes[i] / total) for i in range(3))
            for k in global_state
        }

        vy, vp = evaluate_state(global_state, val_loader)
        va = accuracy_score(vy, vp)
        print(f"Seed {seed} FedAvg round {rnd+1}: val={va:.4f}")
        if va > best_val:
            best_val, best_f = va, copy.deepcopy(global_state)

    f_time = time.time() - start

    # Evaluate both selected checkpoints on the test set
    for name, state, elapsed in [
        ("Centralized ViT", best_c, c_time),
        ("Federated FedAvg", best_f, f_time)
    ]:
        y, p = evaluate_state(state, test_loader)
        all_results.append({
            "seed": seed,
            "model": name,
            "test_accuracy": accuracy_score(y, p),
            "precision_class1": precision_score(y, p, zero_division=0),
            "recall_class1": recall_score(y, p, zero_division=0),
            "f1_class1": f1_score(y, p, zero_division=0),
            "time_seconds": elapsed
        })

    print(f"Finished seed {seed}")

for seed in [7, 21]:
    run_trial(seed)

df = pd.DataFrame(all_results)
df.to_csv("/content/reproducibility_results.csv", index=False)

print("\n--- Results for each additional run ---")
print(df.to_string(index=False))

print("\n--- Mean and standard deviation across the two new runs ---")
print(
    df.groupby("model")[
        ["test_accuracy", "precision_class1",
         "recall_class1", "f1_class1"]
    ].agg(["mean", "std"]).round(4).to_string()
)

print("\nSaved: /content/reproducibility_results.csv")

Seed 7 centralized epoch 1: val=0.9351
Seed 7 centralized epoch 2: val=0.9828
Seed 7 FedAvg round 1: val=0.9618
Seed 7 FedAvg round 2: val=0.9733
Finished seed 7
Seed 21 centralized epoch 1: val=0.9714
Seed 21 centralized epoch 2: val=0.9752
Seed 21 FedAvg round 1: val=0.9618
Seed 21 FedAvg round 2: val=0.9733
Finished seed 21

--- Results for each additional run ---
 seed            model  test_accuracy  precision_class1  recall_class1  f1_class1  time_seconds
    7  Centralized ViT       0.887821          0.849345       0.997436   0.917453    376.142489
    7 Federated FedAvg       0.902244          0.874715       0.984615   0.926417    387.158865
   21  Centralized ViT       0.868590          0.829060       0.994872   0.904429    378.234316
   21 Federated FedAvg       0.911859          0.883295       0.989744   0.933495    388.115728

--- Mean and standard deviation across the two new runs ---
                 test_accuracy         precision_class1         recall_class1         f1_

In [ ]:

# Display the complete results from both additional runs
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print(df.to_string(index=False))

print("\n--- Mean and standard deviation ---")
summary_stats = df.groupby("model").agg(
    runs=("seed", "count"),
    accuracy_mean=("test_accuracy", "mean"),
    accuracy_std=("test_accuracy", "std"),
    precision_mean=("precision_class1", "mean"),
    recall_mean=("recall_class1", "mean"),
    f1_mean=("f1_class1", "mean"),
    time_mean=("time_seconds", "mean")
)

print(summary_stats.round(4).to_string())

# Download the complete results
from google.colab import files
files.download("/content/reproducibility_results.csv")

 seed            model  test_accuracy  precision_class1  recall_class1  f1_class1  time_seconds
    7  Centralized ViT       0.887821          0.849345       0.997436   0.917453    376.142489
    7 Federated FedAvg       0.902244          0.874715       0.984615   0.926417    387.158865
   21  Centralized ViT       0.868590          0.829060       0.994872   0.904429    378.234316
   21 Federated FedAvg       0.911859          0.883295       0.989744   0.933495    388.115728

--- Mean and standard deviation ---
                  runs  accuracy_mean  accuracy_std  precision_mean  recall_mean  f1_mean  time_mean
model                                                                                               
Centralized ViT      2         0.8782        0.0136          0.8392       0.9962   0.9109   377.1884
Federated FedAvg     2         0.9071        0.0068          0.8790       0.9872   0.9300   387.6373


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import pandas as pd
import os

# Load saved results if df is not available
if "df" not in globals():
    if os.path.exists("/content/reproducibility_results.csv"):
        df = pd.read_csv("/content/reproducibility_results.csv")
    else:
        print("Results CSV nahi mili. Experiment cell dobara run karna hoga.")

if "df" in globals():
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 220)

    print("COMPLETE EXPERIMENT RESULTS")
    print(df.to_string(index=False))

    print("\nMEAN AND STANDARD DEVIATION")
    summary_stats = df.groupby("model").agg(
        runs=("seed", "count"),
        accuracy_mean=("test_accuracy", "mean"),
        accuracy_std=("test_accuracy", "std"),
        precision_mean=("precision_class1", "mean"),
        recall_mean=("recall_class1", "mean"),
        f1_mean=("f1_class1", "mean"),
        time_mean=("time_seconds", "mean")
    )

    print(summary_stats.round(4).to_string())

COMPLETE EXPERIMENT RESULTS
 seed            model  test_accuracy  precision_class1  recall_class1  f1_class1  time_seconds
    7  Centralized ViT       0.887821          0.849345       0.997436   0.917453    376.142489
    7 Federated FedAvg       0.902244          0.874715       0.984615   0.926417    387.158865
   21  Centralized ViT       0.868590          0.829060       0.994872   0.904429    378.234316
   21 Federated FedAvg       0.911859          0.883295       0.989744   0.933495    388.115728

MEAN AND STANDARD DEVIATION
                  runs  accuracy_mean  accuracy_std  precision_mean  recall_mean  f1_mean  time_mean
model                                                                                               
Centralized ViT      2         0.8782        0.0136          0.8392       0.9962   0.9109   377.1884
Federated FedAvg     2         0.9071        0.0068          0.8790       0.9872   0.9300   387.6373
